In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import os

class DendrimerJSONLoader:
    def __init__(self):
        self.core_data = None
        self.branch_parts_data = []
        self.core_name = None
        self.branch_names = []

        # Контейнеры для организации интерфейса
        self.core_container = widgets.VBox()
        self.branch_container = widgets.VBox()
        self.status_container = widgets.VBox()
        self.validation_container = widgets.VBox()

        self.initialize_core_section()
        self.initialize_branch_section()
        self.initialize_status_section()
        self.initialize_validation_section()

        # Главный контейнер
        self.main_container = widgets.VBox([
            widgets.HTML("<h2>🎯 Загрузка компонентов дендримера</h2>"),
            self.core_container,
            self.branch_container,
            self.status_container,
            self.validation_container
        ])

    def initialize_core_section(self):
        """Инициализация секции Core"""
        self.core_input = widgets.Text(
            placeholder='Введите путь к JSON файлу для Core...',
            layout=widgets.Layout(width='400px')
        )
        self.core_load_btn = widgets.Button(description='Загрузить Core')
        self.core_name_dropdown = widgets.Dropdown(
            options=[],
            description='Имя:',
            disabled=True,
            layout=widgets.Layout(width='300px')
        )
        self.core_status = widgets.Output()

        self.core_load_btn.on_click(self.load_core)
        self.core_name_dropdown.observe(self.on_core_name_change, names='value')

        self.core_container.children = [
            widgets.HTML("<h3>🎯 Core Component</h3>"),
            widgets.HBox([self.core_input, self.core_load_btn]),
            widgets.HBox([self.core_name_dropdown]),
            self.core_status
        ]

    def initialize_branch_section(self):
        """Инициализация секции Branch Parts"""
        self.branch_inputs = []
        self.branch_load_btns = []
        self.branch_name_dropdowns = []
        self.branch_statuses = []

        # Создаем начальный Branch Part
        self.create_branch_part()

        self.add_branch_btn = widgets.Button(
            description='+ Добавить Branch Part',
            button_style='success',
            layout=widgets.Layout(width='200px')
        )
        self.add_branch_btn.on_click(self.add_branch_part)

        self.update_branch_display()

    def initialize_status_section(self):
        """Инициализация секции статуса"""
        self.overall_status = widgets.Output()
        self.status_container.children = [
            widgets.HTML("<h3>📈 Общий статус</h3>"),
            self.overall_status
        ]

    def initialize_validation_section(self):
        """Инициализация секции проверки"""
        self.validate_btn = widgets.Button(
            description='🔍 Проверить данные',
            button_style='info',
            layout=widgets.Layout(width='200px')
        )
        self.validate_output = widgets.Output()

        self.validate_btn.on_click(self.validate_loaded_data)

        self.validation_container.children = [
            widgets.HTML("<h3>🔍 Проверка данных</h3>"),
            self.validate_btn,
            self.validate_output
        ]

    def create_branch_part(self, index=None):
        """Создание нового Branch Part компонента"""
        if index is None:
            index = len(self.branch_inputs)

        branch_input = widgets.Text(
            placeholder=f'Введите путь к JSON файлу для Branch Part {index + 1}...',
            layout=widgets.Layout(width='400px')
        )

        branch_load_btn = widgets.Button(
            description=f'Загрузить Branch {index + 1}',
            layout=widgets.Layout(width='150px')
        )

        branch_name_dropdown = widgets.Dropdown(
            options=[],
            description='Имя:',
            disabled=True,
            layout=widgets.Layout(width='300px')
        )

        branch_status = widgets.Output()

        # Обработчики событий
        def make_load_handler(idx):
            def load_handler(btn):
                self.load_branch_part(idx)
            return load_handler

        def make_name_handler(idx):
            def name_handler(change):
                self.on_branch_name_change(idx, change)
            return name_handler

        branch_load_btn.on_click(make_load_handler(index))
        branch_name_dropdown.observe(make_name_handler(index), names='value')

        self.branch_inputs.append(branch_input)
        self.branch_load_btns.append(branch_load_btn)
        self.branch_name_dropdowns.append(branch_name_dropdown)
        self.branch_statuses.append(branch_status)

        return branch_input, branch_load_btn, branch_name_dropdown, branch_status

    def update_branch_display(self):
        """Обновление отображения Branch Parts"""
        branch_sections = []

        for i, (branch_input, branch_btn, branch_dropdown, branch_status) in enumerate(zip(
            self.branch_inputs, self.branch_load_btns, self.branch_name_dropdowns, self.branch_statuses)):

            branch_section = widgets.VBox([
                widgets.HTML(f"<b>🌿 Branch Part {i + 1}</b>"),
                widgets.HBox([branch_input, branch_btn]),
                widgets.HBox([branch_dropdown]),
                branch_status
            ])
            branch_sections.append(branch_section)

        # Добавляем кнопку добавления в конец
        branch_sections.append(
            widgets.HBox([self.add_branch_btn], layout=widgets.Layout(margin='20px 0px 0px 0px'))
        )

        self.branch_container.children = [
            widgets.HTML("<h3>🌿 Branch Parts</h3>"),
            *branch_sections
        ]

    def load_core(self, btn):
        """Загрузка Core JSON файла"""
        with self.core_status:
            clear_output()
            filepath = self.core_input.value.strip()

            if not filepath:
                print("❌ Введите путь к файлу")
                return

            if not os.path.exists(filepath):
                print(f"❌ Файл не найден: {filepath}")
                return

            try:
                with open(filepath, 'r', encoding='utf-8') as f:
                    loaded_data = json.load(f)

                if isinstance(loaded_data, list):
                    names = [item.get('name', f'component_{i}') for i, item in enumerate(loaded_data)]
                    self.core_name_dropdown.options = names
                    self.core_name_dropdown.disabled = False
                    if names:
                        self.core_name_dropdown.value = names[0]
                    self.core_data = loaded_data
                    print(f"✅ Core загружен: {filepath}")
                    print(f"📊 Найдено компонентов: {len(loaded_data)}")

                elif isinstance(loaded_data, dict):
                    name = loaded_data.get('name', 'core_component')
                    self.core_name_dropdown.options = [name]
                    self.core_name_dropdown.value = name
                    self.core_name_dropdown.disabled = False
                    self.core_data = [loaded_data]
                    self.core_name = name
                    print(f"✅ Core загружен: {filepath}")
                    print(f"📊 Компонент: {name}")

                else:
                    print("❌ Неверный формат JSON файла")

                self.update_overall_status()

            except Exception as e:
                print(f"❌ Ошибка загрузки: {e}")

    def on_core_name_change(self, change):
        """Обработка изменения выбора имени Core"""
        if self.core_data and change['new'] is not None:
            selected_name = change['new']
            self.core_name = selected_name
            with self.core_status:
                clear_output()
                print(f"✅ Выбран Core: {selected_name}")
            self.update_overall_status()

    def load_branch_part(self, index):
        """Загрузка Branch Part JSON файла"""
        if index >= len(self.branch_inputs):
            return

        with self.branch_statuses[index]:
            clear_output()
            filepath = self.branch_inputs[index].value.strip()

            if not filepath:
                print("❌ Введите путь к файлу")
                return

            if not os.path.exists(filepath):
                print(f"❌ Файл не найден: {filepath}")
                return

            try:
                with open(filepath, 'r', encoding='utf-8') as f:
                    loaded_data = json.load(f)

                branch_name_dropdown = self.branch_name_dropdowns[index]

                if isinstance(loaded_data, list):
                    names = [item.get('name', f'component_{i}') for i, item in enumerate(loaded_data)]
                    branch_name_dropdown.options = names
                    branch_name_dropdown.disabled = False
                    if names:
                        branch_name_dropdown.value = names[0]

                    if index < len(self.branch_parts_data):
                        self.branch_parts_data[index] = loaded_data
                    else:
                        self.branch_parts_data.append(loaded_data)

                    print(f"✅ Branch Part {index + 1} загружен: {filepath}")
                    print(f"📊 Найдено компонентов: {len(loaded_data)}")

                elif isinstance(loaded_data, dict):
                    name = loaded_data.get('name', f'branch_{index}')
                    branch_name_dropdown.options = [name]
                    branch_name_dropdown.value = name
                    branch_name_dropdown.disabled = False

                    if index < len(self.branch_parts_data):
                        self.branch_parts_data[index] = [loaded_data]
                    else:
                        self.branch_parts_data.append([loaded_data])

                    if index < len(self.branch_names):
                        self.branch_names[index] = name
                    else:
                        self.branch_names.append(name)

                    print(f"✅ Branch Part {index + 1} загружен: {filepath}")
                    print(f"📊 Компонент: {name}")

                else:
                    print("❌ Неверный формат JSON файла")
                    return

                self.update_overall_status()

            except Exception as e:
                print(f"❌ Ошибка загрузки: {e}")

    def on_branch_name_change(self, index, change):
        """Обработка изменения выбора имени Branch Part"""
        if (index < len(self.branch_parts_data) and
            self.branch_parts_data[index] and
            change['new'] is not None):

            selected_name = change['new']
            if index < len(self.branch_names):
                self.branch_names[index] = selected_name
            else:
                self.branch_names.append(selected_name)

            with self.branch_statuses[index]:
                clear_output()
                print(f"✅ Выбран Branch Part: {selected_name}")
            self.update_overall_status()

    def add_branch_part(self, btn):
        """Добавление нового Branch Part"""
        # Создаем новый Branch Part
        self.create_branch_part()

        # Обновляем только отображение Branch Parts
        self.update_branch_display()

        # Обновляем общий статус
        self.update_overall_status()

    def update_overall_status(self):
        """Обновление общего статуса загрузки"""
        with self.overall_status:
            clear_output()
            print("📊 СТАТУС ЗАГРУЗКИ:")
            print("=" * 50)
            core_status = f"✅ {self.core_name}" if self.core_name else "❌ Не выбран"
            print(f"Core: {core_status}")

            for i in range(len(self.branch_inputs)):
                if i < len(self.branch_names) and self.branch_names[i]:
                    status = f"✅ {self.branch_names[i]}"
                else:
                    status = "❌ Не выбран"
                print(f"Branch Part {i + 1}: {status}")
            print("=" * 50)

            if self.core_name and any(self.branch_names):
                print("🎯 Все готово для сборки дендримеров!")

    def validate_loaded_data(self, btn):
        """Проверка корректности загруженных данных"""
        data = self.get_loaded_data()

        with self.validate_output:
            clear_output()

            if not data['core']:
                print("❌ Core компонент не выбран")
                return False

            if len(data['branch_parts']) == 0:
                print("❌ Нет выбранных Branch Parts")
                return False

            print("✅ Все данные загружены корректно")
            print(f"🎯 Core: {data['core_name']}")
            print(f"🌿 Branch Parts: {len(data['branch_parts'])}")

            for i, branch in enumerate(data['branch_parts']):
                print(f"  Branch {i + 1}: {data['branch_names'][i]}")

            return True

    def get_loaded_data(self):
        """Возвращает загруженные данные с выбранными именами"""
        selected_core = None
        selected_branches = []

        if self.core_data and self.core_name:
            for component in self.core_data:
                if component.get('name') == self.core_name:
                    selected_core = component
                    break

        for i, branch_data in enumerate(self.branch_parts_data):
            if i < len(self.branch_names) and self.branch_names[i] and branch_data:
                for component in branch_data:
                    if component.get('name') == self.branch_names[i]:
                        selected_branches.append(component)
                        break

        return {
            'core': selected_core,
            'branch_parts': selected_branches,
            'core_name': self.core_name,
            'branch_names': self.branch_names.copy()
        }

    def display(self):
        """Отображение интерфейса"""
        display(self.main_container)


In [2]:
# Создание и отображение интерфейса
json_loader = DendrimerJSONLoader()
json_loader.display()


In [3]:
def get_loaded_json_data():
    """Функция для получения всех загруженных данных с выбранными именами"""
    return json_loader.get_loaded_data()


In [4]:
get_loaded_json_data()

{'core': None, 'branch_parts': [], 'core_name': None, 'branch_names': []}

### Сохранение и загрузка данных дендримеров


In [5]:
def save_dendrimer_data(filename='dendrimer_components.json'):
    """Сохранение данных дендримера в JSON файл"""
    data = json_loader.get_loaded_data()

    # Конвертация в JSON-совместимый формат
    serializable_data = {
        'core': data['core'],
        'branch_parts': data['branch_parts'],
        'core_name': data['core_name'],
        'branch_names': data['branch_names']
    }

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(serializable_data, f, indent=2, ensure_ascii=False)

    print(f"✅ Данные дендримера сохранены в файл: {filename}")
    print(f"🎯 Core: {data['core_name']}")
    print(f"🌿 Branch Parts: {len(data['branch_parts'])}")

def load_dendrimer_data(filename='dendrimer_components.json'):
    """Загрузка данных дендримера из JSON файла"""
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            loaded_data = json.load(f)

        # Восстанавливаем данные в загрузчик
        json_loader.core_data = [loaded_data['core']] if loaded_data['core'] else []
        json_loader.core_name = loaded_data['core_name']
        json_loader.branch_parts_data = [[branch] for branch in loaded_data['branch_parts']]
        json_loader.branch_names = loaded_data['branch_names']

        # Обновляем интерфейс
        if json_loader.core_data:
            core_component = json_loader.core_data[0]
            json_loader.core_name_dropdown.options = [json_loader.core_name]
            json_loader.core_name_dropdown.value = json_loader.core_name
            json_loader.core_name_dropdown.disabled = False
            with json_loader.core_status:
                clear_output()
                print(f"✅ Core загружен из файла: {json_loader.core_name}")

        # Обновляем Branch Parts
        json_loader.branch_inputs = []
        json_loader.branch_load_btns = []
        json_loader.branch_name_dropdowns = []
        json_loader.branch_statuses = []

        for i, branch_name in enumerate(json_loader.branch_names):
            branch_input, branch_btn, branch_dropdown, branch_status = json_loader.create_branch_part(i)

            # Устанавливаем значения для dropdown
            if i < len(json_loader.branch_parts_data) and json_loader.branch_parts_data[i]:
                branch_component = json_loader.branch_parts_data[i][0]
                branch_dropdown.options = [branch_name]
                branch_dropdown.value = branch_name
                branch_dropdown.disabled = False

                with branch_status:
                    print(f"✅ Branch Part {i+1} загружен из файла: {branch_name}")

        # Обновляем отображение
        json_loader.update_branch_display()
        json_loader.update_overall_status()

        print(f"✅ Данные загружены из файла: {filename}")
        print(f"🎯 Core: {json_loader.core_name}")
        print(f"🌿 Branch Parts: {len(json_loader.branch_parts_data)}")

    except FileNotFoundError:
        print(f"❌ Файл {filename} не найден")
    except Exception as e:
        print(f"❌ Ошибка загрузки: {e}")


### Виджеты для управления сохранением и загрузкой


In [6]:
# Виджеты для управления данными
save_btn = widgets.Button(
    description='💾 Сохранить данные',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

load_btn = widgets.Button(
    description='📂 Загрузить данные',
    button_style='info',
    layout=widgets.Layout(width='200px')
)

filename_input = widgets.Text(
    value='dendrimer_components.json',
    placeholder='Имя файла',
    layout=widgets.Layout(width='250px')
)

file_status = widgets.Output()

def on_save_clicked(btn):
    with file_status:
        clear_output()
        save_dendrimer_data(filename_input.value)

def on_load_clicked(btn):
    with file_status:
        clear_output()
        load_dendrimer_data(filename_input.value)

save_btn.on_click(on_save_clicked)
load_btn.on_click(on_load_clicked)

# Отображение виджетов управления
display(widgets.HTML("<h3>💾 Управление данными</h3>"))
display(widgets.HBox([filename_input, save_btn, load_btn]))
display(file_status)


HTML(value='<h3>💾 Управление данными</h3>')

Output()

In [7]:
from rdkit import Chem
from rdkit.Chem import Draw, AllChem
import ipywidgets as widgets
from IPython.display import display, clear_output

class AtomIndexTracker:
    """Класс для отслеживания и обновления индексов атомов"""

    def __init__(self, mol):
        self.original_mol = mol
        self.current_mol = Chem.Mol(mol)
        self.index_map = {i: i for i in range(mol.GetNumAtoms())}
        self.reverse_map = {i: i for i in range(mol.GetNumAtoms())}

    def get_current_index(self, original_idx):
        """Получить текущий индекс для исходного атома"""
        return self.index_map.get(original_idx, original_idx)

    def get_original_index(self, current_idx):
        """Получить исходный индекс для текущего атома"""
        return self.reverse_map.get(current_idx, current_idx)

    def update_after_deletion(self, deleted_atoms):
        """
        Обновить карту индексов после удаления атомов
        deleted_atoms: список индексов удаленных атомов в ТЕКУЩЕЙ молекуле
        """
        if not deleted_atoms:
            return

        # Сортируем удаленные атомы в порядке убывания
        deleted_atoms_sorted = sorted(deleted_atoms, reverse=True)

        # Создаем новые карты
        new_index_map = {}
        new_reverse_map = {}

        current_to_new = {}
        new_idx = 0

        # Строим отображение текущих индексов на новые
        for current_idx in range(self.current_mol.GetNumAtoms()):
            if current_idx not in deleted_atoms:
                current_to_new[current_idx] = new_idx
                new_idx += 1

        # Обновляем основную карту
        for orig_idx, current_idx in self.index_map.items():
            if current_idx in current_to_new:
                new_current_idx = current_to_new[current_idx]
                new_index_map[orig_idx] = new_current_idx
                new_reverse_map[new_current_idx] = orig_idx

        self.index_map = new_index_map
        self.reverse_map = new_reverse_map

    def get_current_molecule(self):
        """Получить текущую молекулу"""
        return self.current_mol

    def set_current_molecule(self, mol):
        """Установить текущую молекулу и обновить карты"""
        self.current_mol = Chem.Mol(mol)
        # После серьезных изменений перестраиваем карты
        self._rebuild_maps()

    def _rebuild_maps(self):
        """Перестроить карты индексов (используется после major изменений)"""
        # Этот метод может быть сложным, поэтому в упрощенной версии
        # мы будем считать, что основные атомы сохраняют свои позиции
        pass

class TrackedReplacementGroupRemover:
    """Улучшенный удалятель групп с отслеживанием индексов"""

    def __init__(self):
        pass

    def find_replacement_group(self, mol, start_atom_idx, target_smiles):
        """
        Находит атомы, составляющие целевую группу, присоединенную к стартовому атому
        Возвращает список атомных индексов группы и индекс связи для разрыва
        """
        mol_copy = Chem.Mol(mol)

        try:
            target_mol = Chem.MolFromSmiles(target_smiles)
            if target_mol is None:
                print(f"❌ Неверные SMILES для группы: {target_smiles}")
                return None, None

            start_atom = mol_copy.GetAtomWithIdx(start_atom_idx)
            neighbors = [x.GetIdx() for x in start_atom.GetNeighbors()]

            matches = mol_copy.GetSubstructMatches(target_mol)

            for match in matches:
                if start_atom_idx in match:
                    group_atoms = list(match)
                    group_atoms.remove(start_atom_idx)

                    for atom_idx in group_atoms:
                        bond = mol_copy.GetBondBetweenAtoms(start_atom_idx, atom_idx)
                        if bond is not None:
                            return group_atoms, bond.GetIdx()

            return None, None

        except Exception as e:
            print(f"❌ Ошибка при поиске группы {target_smiles}: {e}")
            return None, None

    def remove_replacement_group_with_tracking(self, tracker, atom_idx, replacement_group):
        """
        Удаление замещающей группы с отслеживанием индексов
        """
        mol = tracker.get_current_molecule()

        if replacement_group == "H":
            # Удаляем водород
            atom = mol.GetAtomWithIdx(atom_idx)
            current_hs = atom.GetNumExplicitHs()
            atom.SetNumExplicitHs(max(0, current_hs - 1))
            print(f"  🔴 Удален H от атома {atom_idx}")
            # Для водорода не меняем карту индексов
            return mol, atom_idx

        elif replacement_group == "":
            # Пустая строка - изменяем кратность связи
            return self._handle_empty_replacement_with_tracking(tracker, atom_idx)

        else:
            # Сложная группа - ищем и удаляем всю группу
            return self._handle_complex_replacement_with_tracking(tracker, atom_idx, replacement_group)

    def _handle_empty_replacement_with_tracking(self, tracker, atom_idx):
        """Обработка пустой replacement_group с отслеживанием"""
        mol = tracker.get_current_molecule()
        atom = mol.GetAtomWithIdx(atom_idx)

        for bond in atom.GetBonds():
            if bond.GetBeginAtomIdx() == atom_idx or bond.GetEndAtomIdx() == atom_idx:
                bond_type = bond.GetBondType()

                if bond_type == Chem.BondType.TRIPLE:
                    bond.SetBondType(Chem.BondType.DOUBLE)
                    print(f"  🔄 Тройная → двойная связь для атома {atom_idx}")
                    break
                elif bond_type == Chem.BondType.DOUBLE:
                    bond.SetBondType(Chem.BondType.SINGLE)
                    print(f"  🔄 Двойная → одинарная связь для атома {atom_idx}")
                    break
                elif bond_type == Chem.BondType.SINGLE:
                    atom.SetNumExplicitHs(atom.GetNumExplicitHs() + 1)
                    print(f"  🔄 Добавлен H к одинарной связи атома {atom_idx}")
                    break

        return mol, atom_idx

    def _handle_complex_replacement_with_tracking(self, tracker, atom_idx, replacement_group):
        """Обработка сложных замещающих групп с отслеживанием индексов"""
        print(f"  🔍 Поиск группы '{replacement_group}' у атома {atom_idx}")

        mol = tracker.get_current_molecule()
        group_atoms, bond_idx = self.find_replacement_group(mol, atom_idx, replacement_group)

        if group_atoms is None or bond_idx is None:
            print(f"  ⚠️ Группа '{replacement_group}' не найдена у атома {atom_idx}, удаляем H")
            atom = mol.GetAtomWithIdx(atom_idx)
            atom.SetNumExplicitHs(max(0, atom.GetNumExplicitHs() - 1))
            return mol, atom_idx

        # Создаем редактируемую молекулу для удаления группы
        editor = Chem.EditableMol(mol)

        try:
            # Удаляем связь с группой
            bond = mol.GetBondWithIdx(bond_idx)
            editor.RemoveBond(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())

            # Удаляем атомы группы (в обратном порядке)
            group_atoms_sorted = sorted(group_atoms, reverse=True)
            for atom_to_remove in group_atoms_sorted:
                editor.RemoveAtom(atom_to_remove)

            result_mol = editor.GetMol()

            # ОБНОВЛЯЕМ ТРЕКЕР после удаления атомов
            tracker.update_after_deletion(group_atoms_sorted)
            tracker.set_current_molecule(result_mol)

            print(f"  ✅ Удалена группа '{replacement_group}' от атома {atom_idx}")
            print(f"  📊 Удалено атомов: {len(group_atoms_sorted)}")

            # Возвращаем тот же atom_idx, так как стартовый атом не удалялся
            return result_mol, atom_idx

        except Exception as e:
            print(f"  ❌ Ошибка при удалении группы '{replacement_group}': {e}")
            return mol, atom_idx


In [8]:
class SmartDendrimerBuilder:
    def __init__(self):
        self.generations = []
        self.current_molecule = None
        self.connection_points = []  # Храним исходные индексы
        self.group_remover = TrackedReplacementGroupRemover()
        self.debug_output = widgets.Output()
        self.atom_tracker = None

    def draw_molecule_with_atom_indices(self, mol, size=(600, 400), title=""):
        """Рисует молекулу с подписями индексов атомов"""
        mol_for_drawing = Chem.Mol(mol)

        for atom in mol_for_drawing.GetAtoms():
            atom.SetProp('atomNote', str(atom.GetIdx()))

        img = Draw.MolToImage(mol_for_drawing, size=size, kekulize=True)

        with self.debug_output:
            if title:
                print(f"🎨 {title}")
            display(img)
            print(f"📋 SMILES: {Chem.MolToSmiles(mol_for_drawing)}")
            print(f"🔢 Всего атомов: {mol_for_drawing.GetNumAtoms()}")
            print("-" * 50)

        return img

    def prepare_core_connections(self, core_data):
        """Подготовка начальных точек роста из Core"""
        connection_atoms = core_data['connection_atoms'][1:]
        replacement_groups = core_data['replacement_groups'][1:]

        while len(replacement_groups) < len(connection_atoms):
            replacement_groups.append("H")

        return list(zip(connection_atoms, replacement_groups))

    def prepare_branch_connections(self, branch_data):
        """Подготовка точек подключения для Branch Part"""
        if not branch_data['connection_atoms']:
            return None, []

        connect_atom = branch_data['connection_atoms'][0]
        connect_replacement = branch_data['replacement_groups'][0] if branch_data['replacement_groups'] else "H"

        growth_atoms = branch_data['connection_atoms'][1:]
        growth_replacements = branch_data['replacement_groups'][1:] if len(branch_data['replacement_groups']) > 1 else []

        while len(growth_replacements) < len(growth_atoms):
            growth_replacements.append("H")

        new_growth_points = list(zip(growth_atoms, growth_replacements))
        return (connect_atom, connect_replacement), new_growth_points

    def remove_replacement_group(self, tracker, atom_idx, replacement_group):
        """Удаление замещаемой группы с использованием трекера"""
        with self.debug_output:
            print(f"  🗑️ Удаление группы '{replacement_group}' от атома {atom_idx}")

        return self.group_remover.remove_replacement_group_with_tracking(
            tracker, atom_idx, replacement_group
        )

    def connect_molecules(self, mol1, mol2, atom1_idx, atom2_idx):
        """Соединение двух молекул через указанные атомы"""
        with self.debug_output:
            print(f"  🔗 Соединяем атом {atom1_idx} (из {mol1.GetNumAtoms()} атомов) с атомом {atom2_idx} (из {mol2.GetNumAtoms()} атомов)")

        combined = Chem.CombineMols(mol1, mol2)
        editor = Chem.EditableMol(combined)

        num_atoms_mol1 = mol1.GetNumAtoms()
        new_atom2_idx = atom2_idx + num_atoms_mol1

        editor.AddBond(atom1_idx, new_atom2_idx, Chem.BondType.SINGLE)
        result = editor.GetMol()

        try:
            result = Chem.RemoveHs(result)
            AllChem.Compute2DCoords(result)
        except Exception as e:
            with self.debug_output:
                print(f"  ⚠️ Ошибка при обработке молекулы: {e}")

        return result

    def build_generation(self, core_data, branch_parts_data, generation_num=1):
        """Построение поколения с умным отслеживанием индексов"""
        with self.debug_output:
            print(f"\n🔨 СТРОИМ ПОКОЛЕНИЕ {generation_num}")
            print("=" * 60)

        if self.current_molecule is None:
            # Инициализация с Core
            core_smiles = core_data['smiles']
            self.current_molecule = Chem.MolFromSmiles(core_smiles)
            self.atom_tracker = AtomIndexTracker(self.current_molecule)

            with self.debug_output:
                print(f"🎯 Инициализация Core: {core_data['name']}")
                print(f"🎯 Core SMILES: {core_smiles}")

            self.draw_molecule_with_atom_indices(
                self.current_molecule,
                title=f"Core молекула - {core_data['name']}"
            )

            core_growth_points = self.prepare_core_connections(core_data)
            self.connection_points = []

            for atom_idx, replacement in core_growth_points:
                # Сохраняем исходные индексы
                self.connection_points.append((self.current_molecule, atom_idx, replacement))

            with self.debug_output:
                print(f"📌 Начальные точки роста из Core: {len(self.connection_points)}")
                for i, (_, atom_idx, repl) in enumerate(self.connection_points):
                    print(f"    Точка {i+1}: атом {atom_idx}, замена '{repl}'")

        # Определяем Branch Part
        if generation_num - 1 < len(branch_parts_data):
            branch_data = branch_parts_data[generation_num - 1]
        else:
            branch_data = branch_parts_data[-1]

        with self.debug_output:
            print(f"🌿 Используем Branch Part: {branch_data['name']}")
            print(f"🌿 Branch SMILES: {branch_data['smiles']}")

        # Подготавливаем данные Branch Part
        (connect_atom, connect_replacement), new_growth_points = self.prepare_branch_connections(branch_data)

        with self.debug_output:
            print(f"🔌 Подключение: атом {connect_atom}, замена '{connect_replacement}'")
            print(f"🌱 Новые точки роста: {len(new_growth_points)}")

        # Визуализируем исходное состояние
        self.draw_molecule_with_atom_indices(
            self.current_molecule,
            title=f"Исходная молекула поколения {generation_num}"
        )

        # Пошаговое построение с отслеживанием индексов
        working_molecule = Chem.Mol(self.current_molecule)
        working_tracker = AtomIndexTracker(working_molecule)
        new_connection_points = []

        with self.debug_output:
            print(f"\n🔄 ПОСТРОЕНИЕ С ОТСЛЕЖИВАНИЕМ ИНДЕКСОВ")

        for point_idx, (_, original_atom_idx, replacement) in enumerate(self.connection_points):
            with self.debug_output:
                print(f"\n📍 ТОЧКА РОСТА {point_idx + 1}/{len(self.connection_points)}")
                print(f"   Исходный атом: {original_atom_idx}, Замена: '{replacement}'")
                print(f"   Текущая молекула: {working_molecule.GetNumAtoms()} атомов")

            # Получаем текущий индекс атома
            current_atom_idx = working_tracker.get_current_index(original_atom_idx)
            with self.debug_output:
                print(f"   Текущий индекс атома: {current_atom_idx}")

            # Создаем Branch Part
            branch_mol = Chem.MolFromSmiles(branch_data['smiles'])
            branch_tracker = AtomIndexTracker(branch_mol)

            # Удаляем группы с отслеживанием индексов
            dendrimer_cleaned, dendrimer_atom = self.remove_replacement_group(
                working_tracker, current_atom_idx, replacement
            )
            branch_cleaned, branch_atom = self.remove_replacement_group(
                branch_tracker, connect_atom, connect_replacement
            )

            # Соединяем молекулы
            connected_molecule = self.connect_molecules(
                dendrimer_cleaned, branch_cleaned, dendrimer_atom, branch_atom
            )

            # Обновляем рабочую молекулу и трекер
            working_molecule = connected_molecule
            working_tracker.set_current_molecule(working_molecule)

            # Добавляем новые точки роста
            for growth_atom, growth_replacement in new_growth_points:
                # Индексы в объединенной молекуле
                new_atom_idx = growth_atom + dendrimer_cleaned.GetNumAtoms()
                new_connection_points.append((working_molecule, new_atom_idx, growth_replacement))

            with self.debug_output:
                print(f"   ✅ Точка роста {point_idx + 1} обработана")
                print(f"   📍 Добавлено новых точек: {len(new_growth_points)}")

        # Обновляем состояние
        self.current_molecule = working_molecule
        self.connection_points = new_connection_points
        self.atom_tracker = working_tracker

        self.generations.append({
            'generation': generation_num,
            'molecule': Chem.Mol(self.current_molecule),
            'smiles': Chem.MolToSmiles(self.current_molecule),
            'connection_points': len(self.connection_points),
            'branch_used': branch_data['name']
        })

        # Финальная визуализация
        self.draw_molecule_with_atom_indices(
            self.current_molecule,
            title=f"ФИНАЛ - Поколение {generation_num}"
        )

        with self.debug_output:
            print(f"\n✅ ПОКОЛЕНИЕ {generation_num} ПОСТРОЕНО")
            print(f"📊 SMILES: {self.generations[-1]['smiles']}")
            print(f"📌 Новые точки роста: {len(self.connection_points)}")
            print("=" * 60)

        return self.current_molecule, self.connection_points


In [9]:
class SmartDendrimerConstructionInterface:
    def __init__(self):
        self.builder = SmartDendrimerBuilder()

        self.generations_input = widgets.IntText(
            value=2,
            description='Поколений:',
            min=1,
            max=5
        )

        self.build_btn = widgets.Button(
            description='🧠 Построить (с отслеживанием)',
            button_style='success'
        )

        self.reset_btn = widgets.Button(
            description='🔄 Сбросить',
            button_style='warning'
        )

        self.construction_output = widgets.Output()
        self.generations_info = widgets.Output()

        self.build_btn.on_click(self.build_dendrimer)
        self.reset_btn.on_click(self.reset_builder)

    def build_dendrimer(self, btn):
        """Построение дендримера с умным отслеживанием индексов"""
        with self.construction_output:
            clear_output()

            try:
                data = get_loaded_json_data()
            except:
                print("❌ Сначала загрузите данные компонентов")
                return

            if not data['core'] or not data['branch_parts']:
                print("❌ Не все компоненты загружены")
                return

            generations = self.generations_input.value

            print(f"🔨 Начинаем построение дендримера (С ОТСЛЕЖИВАНИЕМ ИНДЕКСОВ)...")
            print(f"🎯 Core: {data['core_name']}")
            print(f"🌿 Branch Parts: {len(data['branch_parts'])}")
            print(f"📈 Поколений: {generations}")
            print("=" * 50)

            for gen in range(1, generations + 1):
                molecule, connections = self.builder.build_generation(
                    data['core'],
                    data['branch_parts'],
                    gen
                )

            self.show_generations_info()
            print("=" * 50)
            print("✅ Построение завершено!")

    def show_generations_info(self):
        """Отображение информации о построенных поколениях"""
        with self.generations_info:
            clear_output()
            print("📊 ИНФОРМАЦИЯ О ПОКОЛЕНИЯХ:")
            print("=" * 60)
            for gen_info in self.builder.generations:
                print(f"Поколение {gen_info['generation']}:")
                print(f"  Branch: {gen_info['branch_used']}")
                print(f"  Точек роста: {gen_info['connection_points']}")
                print(f"  SMILES: {gen_info['smiles']}")
                print("-" * 40)

    def reset_builder(self, btn):
        """Сброс построителя"""
        self.builder = SmartDendrimerBuilder()
        with self.construction_output:
            clear_output()
            print("🔄 Построитель сброшен")
        with self.generations_info:
            clear_output()

    def display(self):
        """Отображение интерфейса"""
        display(widgets.HTML("<h3>🧠 Построение дендримеров (С ОТСЛЕЖИВАНИЕМ ИНДЕКСОВ)</h3>"))
        display(widgets.HBox([self.generations_input, self.build_btn, self.reset_btn]))
        display(self.construction_output)
        display(widgets.HTML("<h4>📊 Отладочная информация</h4>"))
        display(self.builder.debug_output)
        display(widgets.HTML("<h4>📊 Информация о поколениях</h4>"))
        display(self.generations_info)


In [10]:
# Создание и отображение умного интерфейса
smart_construction_interface = SmartDendrimerConstructionInterface()
smart_construction_interface.display()

HTML(value='<h3>🧠 Построение дендримеров (С ОТСЛЕЖИВАНИЕМ ИНДЕКСОВ)</h3>')

Output()

HTML(value='<h4>📊 Отладочная информация</h4>')

Output()

HTML(value='<h4>📊 Информация о поколениях</h4>')

Output()